In [ ]:
!pip install -U scikit-learn-intelex threadpoolctl py-cpuinfo xgboost 2>&1 | tail -n 6

    Uninstalling tbb-2022.3.1:
      Successfully uninstalled tbb-2022.3.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mkl 2025.3.1 requires tbb==2022.*, but you have tbb 2023.2.0 which is incompatible.


In [ ]:
import os, sys, site, glob, importlib

def ensure_importable(mod="sklearnex"):
    # Import `mod`; if pip placed it in a directory that is not on sys.path, add that directory.
    try:
        return importlib.import_module(mod)
    except ModuleNotFoundError:
        roots = glob.glob("/usr/local/lib/python3*/*-packages") + site.getsitepackages() + [site.getusersitepackages()]
        for p in roots:
            if os.path.isdir(os.path.join(p, mod)) and p not in sys.path:
                site.addsitedir(p); print("Added to sys.path:", p)
        importlib.invalidate_caches()
        return importlib.import_module(mod)

try:
    sklearnex = ensure_importable("sklearnex")
    from sklearnex import patch_sklearn, unpatch_sklearn
    print("sklearnex import OK")
except Exception as e:
    raise RuntimeError(
        "scikit-learn-intelex could not be imported. Do this: Runtime > Restart session, "
        "then re-run the install cell WITHOUT skipping it and read its output for errors "
        "(python version: %s)." % sys.version.split()[0]) from e

Added to sys.path: /usr/local/lib/python3.13/site-packages
sklearnex import OK


In [ ]:
import os, time, warnings, logging
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, sklearnex
warnings.filterwarnings("ignore")
print("scikit-learn:", sklearn.__version__, "| scikit-learn-intelex:", getattr(sklearnex, "__version__", "?"))
print("CPUs:", os.cpu_count())
!lscpu | egrep "Model name|^CPU\(s\)"

def timeit(fn, repeat=2, warmup=0):
    for _ in range(warmup): fn()
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter(); out = fn(); best = min(best, time.perf_counter() - t0)
    return best, out

scikit-learn: 1.6.1 | scikit-learn-intelex: 2199.9.9
CPUs: 2
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz


In [ ]:
from sklearnex import patch_sklearn, unpatch_sklearn, get_patch_names
print("Estimators that can be accelerated:\n", get_patch_names())

Estimators that can be accelerated:
 ['sklearn.cluster.DBSCAN', 'sklearn.cluster.KMeans', 'sklearn.decomposition.PCA', 'sklearn.svm.SVR', 'sklearn.svm.SVC', 'sklearn.svm.NuSVR', 'sklearn.svm.NuSVC', 'sklearn.linear_model.LinearRegression', 'sklearn.linear_model.LogisticRegression', 'sklearn.linear_model.Ridge', 'sklearn.linear_model.IncrementalLinearRegression', 'sklearn.manifold.TSNE', 'sklearn.metrics.pairwise_distances', 'sklearn.metrics.roc_auc_score', 'sklearn.model_selection.train_test_split', 'sklearn.neighbors.KNeighborsClassifier', 'sklearn.neighbors.KNeighborsRegressor', 'sklearn.neighbors.NearestNeighbors', 'sklearn.neighbors.LocalOutlierFactor', 'sklearn.ensemble.ExtraTreesClassifier', 'sklearn.ensemble.ExtraTreesRegressor', 'sklearn.ensemble.RandomForestClassifier', 'sklearn.ensemble.RandomForestRegressor', 'sklearn.covariance.IncrementalEmpiricalCovariance', 'sklearn.dummy.DummyRegressor', 'sklearn.ensemble._gb_losses.DummyRegressor', 'sklearn.set_config', 'sklearn.get_co

In [ ]:
# Method A: global
patch_sklearn()
from sklearn.cluster import KMeans
print("A  ->", KMeans.__module__)          # sklearnex.* when patched

# Method B: selective. Recent releases expect the full dotted names listed by get_patch_names(),
# so we resolve short names against that list (works across versions).
def names_for(*short):
    wanted = {s.lower() for s in short}
    return [n for n in get_patch_names() if n.split(".")[-1].lower() in wanted]

unpatch_sklearn()
print("Patching only:", names_for("KMeans", "PCA"))
patch_sklearn(names_for("KMeans", "PCA"))
from sklearn.cluster import KMeans
from sklearn.svm import SVC
print("B  ->", KMeans.__module__, "|", SVC.__module__, "(SVC stays stock)")

# Method C: explicit import
unpatch_sklearn()
from sklearnex.cluster import KMeans as IntelKMeans
print("C  ->", IntelKMeans.__module__)

A  -> sklearnex.cluster.k_means
Patching only: ['sklearn.cluster.KMeans', 'sklearn.decomposition.PCA']
B  -> sklearnex.cluster.k_means | sklearn.svm._classes (SVC stays stock)
C  -> sklearnex.cluster.k_means


Extension for scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)
Extension for scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [ ]:
patch_sklearn()
logging.basicConfig(); logging.getLogger("sklearnex").setLevel(logging.INFO)

from sklearn.cluster import KMeans
from sklearn.svm import SVC
X = np.random.rand(20_000, 10)
KMeans(4, n_init=1).fit(X)                                  # supported
SVC(kernel="precomputed").fit(X[:500] @ X[:500].T, np.arange(500) % 2)   # unsupported kernel -> fallback message expected
logging.getLogger("sklearnex").setLevel(logging.WARNING)

Extension for scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)
INFO:sklearnex: sklearn.cluster.KMeans.fit: running accelerated version on CPU
INFO:sklearnex:sklearn.cluster.KMeans.fit: running accelerated version on CPU
INFO:sklearnex: sklearn.svm.SVC.fit: fallback to original Scikit-learn. All input data transferred to host for further backend computations.
INFO:sklearnex:sklearn.svm.SVC.fit: fallback to original Scikit-learn. All input data transferred to host for further backend computations.


In [ ]:
from sklearn.datasets import make_classification, make_regression, make_blobs

# Scale factor: 1.0 is good for a 2-vCPU Colab VM (~4-6 min total). Use 0.3 for a quick pass, 3+ on a big node.
SCALE = 1.0
rng = np.random.RandomState(42)

Xc, yc = make_classification(int(400_000*SCALE), 40, n_informative=20, random_state=0)
Xr, yr = make_regression(int(400_000*SCALE), 60, noise=5, random_state=0)
Xb, _  = make_blobs(int(600_000*SCALE), 20, centers=8, random_state=0)
Xs, ys = make_classification(int(15_000*SCALE), 30, n_informative=15, random_state=1)     # SVC is O(n^2)
Xd, _  = make_blobs(int(40_000*SCALE), 4, centers=10, cluster_std=0.6, random_state=2)    # DBSCAN
ntest = 20_000
for name in ("Xc","Xr","Xb","Xs","Xd"): print(name, globals()[name].shape)

Xc (400000, 40)
Xr (400000, 60)
Xb (600000, 20)
Xs (15000, 30)
Xd (40000, 4)


In [ ]:
from sklearn.datasets import fetch_covtype
try:
    cov = fetch_covtype(as_frame=False)
    Xall, yall = cov.data.astype(np.float32), cov.target
    src = "Forest CoverType (real)"
except Exception as e:                                     # offline fallback
    Xall, yall = make_classification(581_012, 54, n_informative=30, n_classes=7, random_state=0); Xall = Xall.astype(np.float32); src = "synthetic fallback"
print(src, Xall.shape, "classes:", np.unique(yall))

Forest CoverType (real) (581012, 54) classes: [1 2 3 4 5 6 7]


In [ ]:
def end_to_end(optimized: bool, subsample=300_000):
    from sklearnex import patch_sklearn, unpatch_sklearn
    (patch_sklearn if optimized else unpatch_sklearn)()
    from sklearn.model_selection import train_test_split, RandomizedSearchCV
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import accuracy_score, f1_score, classification_report

    T = {}
    t_all = time.perf_counter()
    X, _, y, _ = train_test_split(Xall, yall, train_size=subsample, stratify=yall, random_state=0)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

    # ---- (1) baseline: linear model
    t0 = time.perf_counter()
    base = Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=200))]).fit(Xtr, ytr)
    T["baseline LR fit"] = time.perf_counter() - t0
    acc_base = accuracy_score(yte, base.predict(Xte))

    # ---- (2) kNN with PCA, hyper-parameter search
    pipe = Pipeline([("sc", StandardScaler()), ("pca", PCA(random_state=0)), ("knn", KNeighborsClassifier())])
    grid = {"pca__n_components": [10, 20, 30], "knn__n_neighbors": [3, 5, 9], "knn__weights": ["uniform", "distance"]}
    t0 = time.perf_counter()
    search = RandomizedSearchCV(pipe, grid, n_iter=6, cv=3, scoring="accuracy", random_state=0, n_jobs=1).fit(Xtr, ytr)
    T["PCA+kNN search"] = time.perf_counter() - t0

    t0 = time.perf_counter(); pred = search.predict(Xte); T["inference"] = time.perf_counter() - t0
    T["TOTAL"] = time.perf_counter() - t_all
    return T, dict(baseline_acc=acc_base, best_acc=accuracy_score(yte, pred), macro_f1=f1_score(yte, pred, average="macro"), best_params=search.best_params_)

T0, M0 = end_to_end(False)
T1, M1 = end_to_end(True)
res = pd.DataFrame({"stock_s": T0, "intel_s": T1}); res["speedup"] = res.stock_s / res.intel_s
display(res.round(2))
print("Stock :", {k: (round(v, 4) if not isinstance(v, dict) else v) for k, v in M0.items()})
print("Intel :", {k: (round(v, 4) if not isinstance(v, dict) else v) for k, v in M1.items()})

Extension for scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


,stock_s,intel_s,speedup
baseline LR fit,44.16,18.17,2.43
PCA+kNN search,1329.89,395.16,3.37
inference,95.61,26.40,3.62
TOTAL,1470.22,440.29,3.34


Stock : {'baseline_acc': 0.7232, 'best_acc': 0.8926, 'macro_f1': 0.8167, 'best_params': {'pca__n_components': 20, 'knn__weights': 'distance', 'knn__n_neighbors': 3}}
Intel : {'baseline_acc': 0.7231, 'best_acc': 0.8926, 'macro_f1': 0.8167, 'best_params': {'pca__n_components': 20, 'knn__weights': 'distance', 'knn__n_neighbors': 3}}


In [ ]:
import time
import numpy as np

print("=" * 66)
print("DEMO: PYTHON LOOPS vs NUMPY BLAS")

N_small = 160
A_list = [[1.0] * N_small for _ in range(N_small)]
B_list = [[2.0] * N_small for _ in range(N_small)]
C_list = [[0.0] * N_small for _ in range(N_small)]
t0 = time.perf_counter()
for i in range(N_small):
    for j in range(N_small):
        s = 0.0
        for k in range(N_small):
            s += A_list[i][k] * B_list[k][j]
        C_list[i][j] = s
t_python = time.perf_counter() - t0
print(f"  Pure Python {N_small}x{N_small}: {t_python:.4f}s")

# same-size comparison (fair), then a big one for GFLOP/s
A_s = np.ones((N_small, N_small)); B_s = np.full((N_small, N_small), 2.0)
_ = A_s @ B_s                                   # warm-up
t0 = time.perf_counter(); C_s = A_s @ B_s; t_np_small = time.perf_counter() - t0
print(f"  NumPy       {N_small}x{N_small}: {t_np_small:.6f}s   -> {t_python / max(t_np_small, 1e-9):,.0f}x faster than the loops")
assert abs(C_s[0, 0] - C_list[0][0]) < 1e-9

N_large = 1024
A_np = np.ones((N_large, N_large), dtype=np.float64)
B_np = np.full((N_large, N_large), 2.0, dtype=np.float64)
_ = np.dot(A_np[:8, :8], B_np[:8, :8])
t0 = time.perf_counter()
C_np = np.dot(A_np, B_np)
t_mkl = time.perf_counter() - t0
gflops = (2.0 * (N_large ** 3) / max(t_mkl, 1e-9)) / 1e9
print(f"  NumPy BLAS  {N_large}x{N_large}: {t_mkl:.4f}s  ({gflops:.2f} GFLOP/s)")
print(f"  Check: C[0,0] = {C_np[0,0]:.1f}  (expect {float(N_large * 2):.1f})")
assert abs(C_np[0, 0] - (N_large * 2.0)) < 1e-6
print("=" * 66)

DEMO: PYTHON LOOPS vs NUMPY BLAS
  Pure Python 160x160: 1.0378s
  NumPy       160x160: 0.001073s   -> 968x faster than the loops
  NumPy BLAS  1024x1024: 0.0960s  (22.37 GFLOP/s)
  Check: C[0,0] = 2048.0  (expect 2048.0)
